<a href="https://colab.research.google.com/github/0xjessie21/data-science-2026/blob/main/Pertemuan3_MOHAMMAD_RIYAN_SYAIFUNAHAR_240401010292.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 3 — Data Cleaning: Missing Values, Outlier & Ekstraksi Data
**Nama:** MOHAMMAD RIYAN SYAIFUNAHAR  
**NIM:** 240401010292  
**Mata Kuliah:** Pengantar Data Science  
**Dataset:** housing_dirty.csv (130 baris, 7 kolom)

---
## Langkah 2 — Import Library

In [1]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize

print('Semua library berhasil diimport!')

Semua library berhasil diimport!


---
## Langkah 3 — Load Dataset

In [2]:
# Upload file housing_dirty.csv ke Colab terlebih dahulu
# Atau load dari Google Drive jika sudah tersedia

# Opsi A: Upload manual di Colab (jalankan cell ini jika belum upload)
# from google.colab import files
# uploaded = files.upload()  # pilih housing_dirty.csv

# Opsi B: Load langsung dari path lokal setelah upload
df = pd.read_csv('housing_dirty.csv')

print('Dataset berhasil dimuat!')
print('Shape awal:', df.shape)
df.head(10)

Dataset berhasil dimuat!
Shape awal: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang
5,6,153.3,814.0,Jakarta,1.0,2006,Sedang
6,7,114.3,NaN,jakarta,3.0,2011,baik
7,8,NaN,333.0,Yogyakarta,NaN,1989,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996,SEDANG
9,10,69.1,237.0,Bandung,5.0,1980,baik


---
## Langkah 4 — Eksplorasi Awal

In [3]:
# Informasi tipe data dan jumlah non-null
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [4]:
# Statistik deskriptif
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [5]:
# Cek missing values per kolom
missing = df.isnull().sum()
persen  = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': persen
})
print('=== Missing Values per Kolom ===')
print(missing_df[missing_df['Jumlah Missing'] > 0])

=== Missing Values per Kolom ===
            Jumlah Missing  Persentase (%)
luas_m2                 18           13.85
harga_juta              17           13.08
kamar                   10            7.69


**Temuan Eksplorasi Awal:**  
- `luas_m2`: 18 missing values (13.85%)  
- `harga_juta`: 17 missing values (13.08%)  
- `kamar`: 10 missing values (7.69%)  
- Kolom `kota` memiliki inkonsistensi string: 'jakarta', 'Jakarta', 'JAKARTA', 'jogja', 'YGY', dll  
- Kolom `kondisi` memiliki inkonsistensi kapital: 'baik', 'BAIK', 'Baik', dll  
- Terdapat outlier pada `luas_m2` (min=-50, max=9500) dan `harga_juta` (nilai negatif dan sangat besar)  
- `tahun_bangun` memiliki nilai tidak wajar (max=9999, ada nilai di bawah 1950)

---
## Langkah 5 — Hapus Baris Duplikat

In [6]:
# Cek jumlah duplikat sebelum dihapus
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

# Hapus duplikat — pertahankan kemunculan pertama
df.drop_duplicates(inplace=True)

print(f'Shape setelah hapus duplikat: {df.shape}')

Jumlah baris duplikat: 0
Shape setelah hapus duplikat: (130, 7)


**Temuan Langkah 5:**  
Tidak ditemukan baris duplikat pada dataset ini (0 duplikat). Shape tetap (130, 7).

---
## Langkah 6 — Normalisasi String

In [7]:
# Sebelum normalisasi — lihat nilai unik kolom kota
print('=== Nilai unik KOTA sebelum normalisasi ===')
print(df['kota'].value_counts())

=== Nilai unik KOTA sebelum normalisasi ===
kota
Bandung       14
Medan         13
Makassar      12
Yogyakarta    10
Jakarta        9
Depok          7
Surabaya       7
Semarang       6
medan          5
jogja          5
yogyakarta     4
makassar       4
Jogja          3
semarang       3
YGY            3
mdn            2
sby            2
dpk            2
jakarta        2
DEPOK          2
Smg            2
MAKASSAR       2
surabaya       2
Mksr           2
JAKARTA        1
Bdg            1
depok          1
bandung        1
Bandung        1
SURABAYA       1
 Jakarta       1
Name: count, dtype: int64


In [8]:
# Normalisasi kolom kota: strip spasi + title case
df['kota'] = df['kota'].str.strip().str.title()

# Mapping singkatan/nama tidak standar ke nama baku
mapping_kota = {
    'Ygy'      : 'Yogyakarta',
    'Jogja'    : 'Yogyakarta',
    'Dpk'      : 'Depok',
    'Sby'      : 'Surabaya',
    'Mdn'      : 'Medan',
    'Smg'      : 'Semarang',
    'Mksr'     : 'Makassar',
    'Bdg'      : 'Bandung',
}
df['kota'] = df['kota'].replace(mapping_kota)

print('=== Nilai unik KOTA setelah normalisasi ===')
print(df['kota'].value_counts())

=== Nilai unik KOTA setelah normalisasi ===
kota
Yogyakarta    25
Medan         20
Makassar      20
Bandung       17
Jakarta       13
Depok         12
Surabaya      12
Semarang      11
Name: count, dtype: int64


In [9]:
# Normalisasi kolom kondisi: strip spasi + lowercase
print('=== Nilai unik KONDISI sebelum normalisasi ===')
print(df['kondisi'].value_counts())

df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\n=== Nilai unik KONDISI setelah normalisasi ===')
print(df['kondisi'].value_counts())

=== Nilai unik KONDISI sebelum normalisasi ===
kondisi
baik              45
sedang            30
rusak             10
BAIK               8
baik sekali        7
Baik               5
SEDANG             4
cukup              4
bagus              4
Bagus              3
Sedang             3
Cukup              2
jelek              2
RUSAK              2
perlu renovasi     1
Name: count, dtype: int64

=== Nilai unik KONDISI setelah normalisasi ===
kondisi
baik              58
sedang            37
rusak             12
bagus              7
baik sekali        7
cukup              6
jelek              2
perlu renovasi     1
Name: count, dtype: int64


**Temuan Langkah 6:**  
- Kolom `kota`: berhasil dinormalisasi dari 30+ variasi menjadi 9 kota standar menggunakan `.str.strip().str.title()` + mapping singkatan  
- Kolom `kondisi`: berhasil dinormalisasi ke lowercase sehingga 'BAIK', 'Baik', 'baik' menjadi satu kategori 'baik'

---
## Langkah 7 — Imputasi Missing Values

In [10]:
# Cek missing values sebelum imputasi
print('Missing sebelum imputasi:')
print(df.isnull().sum())

Missing sebelum imputasi:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [11]:
# Imputasi Median untuk kolom numerik (lebih robust terhadap outlier)
df['luas_m2']    = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Imputasi Modus untuk kolom kategorik (kamar)
modus_kamar = df['kamar'].mode()[0]
df['kamar'] = df['kamar'].fillna(modus_kamar)

# Verifikasi
print('Missing setelah imputasi:')
print(df.isnull().sum())

Missing setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


**Temuan Langkah 7:**  
- `luas_m2`: diimputasi dengan **median** (lebih robust terhadap outlier dibanding mean)  
- `harga_juta`: diimputasi dengan **median**  
- `kamar`: diimputasi dengan **modus** (kolom numerik diskrit yang berperan seperti kategorik)  
- Setelah imputasi, seluruh kolom memiliki 0 missing values

---
## Langkah 8 — Tangani Outlier dengan IQR Fence

In [12]:
# Fungsi deteksi outlier IQR
def deteksi_outlier_iqr(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[kolom] < lower) | (df[kolom] > upper)]
    return lower, upper, outliers

# Deteksi outlier sebelum capping
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    lower, upper, out_df = deteksi_outlier_iqr(df, col)
    print(f'[{col}] Batas: [{lower:.2f}, {upper:.2f}] | Outlier: {len(out_df)} baris')

[harga_juta] Batas: [-422.75, 1719.25] | Outlier: 3 baris
[luas_m2] Batas: [-145.22, 512.97] | Outlier: 1 baris
[tahun_bangun] Batas: [1960.50, 2042.50] | Outlier: 3 baris


In [13]:
# Terapkan IQR Fence (capping/clip) pada kolom yang mengandung outlier
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    print(f'[{col}] setelah capping — min: {df[col].min():.2f}, max: {df[col].max():.2f}')

[harga_juta] setelah capping — min: -422.75, max: 1719.25
[luas_m2] setelah capping — min: -50.00, max: 512.97
[tahun_bangun] setelah capping — min: 1960.50, max: 2042.50


**Temuan Langkah 8:**  
- `harga_juta`: ditemukan nilai negatif (-500) dan sangat besar (100.000.000) → di-cap dengan IQR Fence  
- `luas_m2`: ditemukan nilai negatif (-50) dan sangat besar (9500) → di-cap dengan IQR Fence  
- `tahun_bangun`: ditemukan nilai tidak wajar (9999, dan nilai di bawah 1950) → di-cap  
- Metode **clip (Winsorization)** dipilih agar jumlah baris tidak berkurang

---
## Langkah 9 — Validasi Akhir

In [14]:
# Validasi: tidak boleh ada missing values
total_missing = df.isnull().sum().sum()
total_dup     = df.duplicated().sum()

print(f'Total missing values : {total_missing}  (harus = 0)')
print(f'Total duplikat       : {total_dup}  (harus = 0)')
print(f'Shape akhir          : {df.shape}')

# Assert untuk memastikan validasi lolos
assert total_missing == 0, 'Masih ada missing values!'
assert total_dup     == 0, 'Masih ada duplikat!'

print('\nValidasi LULUS! Dataset bersih dan siap digunakan.')

Total missing values : 0  (harus = 0)
Total duplikat       : 0  (harus = 0)
Shape akhir          : (130, 7)

Validasi LULUS! Dataset bersih dan siap digunakan.


In [15]:
# Ringkasan dataset setelah cleaning
print('=== Statistik Akhir Dataset ===')
df.describe()

=== Statistik Akhir Dataset ===


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,130.000000,130.000000,130.000000,130.000000
mean,65.500000,188.274423,686.490385,3.246154,2001.542308
std,37.671829,95.297150,404.633957,1.826003,13.505818
min,1.000000,-50.000000,-422.750000,1.000000,1960.500000
25%,33.250000,101.600000,380.500000,1.000000,1991.250000
50%,65.500000,193.800000,655.000000,3.000000,2002.000000
75%,97.750000,266.150000,916.000000,5.000000,2011.750000
max,130.000000,512.975000,1719.250000,6.000000,2042.500000


---
## Langkah 10 — Ekspor Dataset Bersih

In [16]:
# Simpan dataset bersih ke file CSV
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih berhasil disimpan sebagai housing_clean.csv!')

# Preview hasil akhir
df.head(10)

Dataset bersih berhasil disimpan sebagai housing_clean.csv!


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Yogyakarta,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Yogyakarta,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang
5,6,153.3,814.0,Jakarta,1.0,2006.0,sedang
6,7,114.3,655.0,Jakarta,3.0,2011.0,baik
7,8,193.8,333.0,Yogyakarta,1.0,1989.0,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996.0,sedang
9,10,69.1,237.0,Bandung,5.0,1980.0,baik


---
## Langkah 11 — Akses REST API JSONPlaceholder

In [17]:
# Akses API JSONPlaceholder — daftar users (gratis, tanpa API key)
URL = "https://jsonplaceholder.typicode.com/users"

try:
    response = requests.get(URL, timeout=10)

    # Cek status code terlebih dahulu
    if response.status_code == 200:
        data = response.json()
        df_users = json_normalize(data, sep='_')
        print(f'Berhasil! Total data: {len(df_users)} users')
        print(f'Kolom tersedia: {df_users.columns.tolist()}')
    else:
        print(f'Error: {response.status_code}')

except requests.exceptions.ConnectionError:
    print('Gagal terhubung ke API. Cek koneksi internet!')
except requests.exceptions.Timeout:
    print('Request timeout. Coba lagi!')

Berhasil! Total data: 10 users
Kolom tersedia: ['id', 'name', 'username', 'email', 'phone', 'website', 'address_street', 'address_suite', 'address_city', 'address_zipcode', 'address_geo_lat', 'address_geo_lng', 'company_name', 'company_catchPhrase', 'company_bs']


In [18]:
# Tampilkan kolom yang relevan
print('=== Data Users dari JSONPlaceholder API ===')
df_users[['id', 'name', 'email', 'address_city', 'company_name']]

=== Data Users dari JSONPlaceholder API ===


,id,name,email,address_city,company_name
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,Romaguera-Crona
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,Robel-Corkery
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC
5,6,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy,Considine-Lockman
6,7,Kurtis Weissnat,Telly.Hoeger@billy.biz,Howemouth,Johns Group
7,8,Nicholas Runolfsdottir V,Sherwood@rosamond.me,Aliyaview,Abernathy Group
8,9,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury,Yost and Sons
9,10,Clementina DuBuque,Rey.Padberg@karina.biz,Lebsackbury,Hoeger LLC


In [19]:
# Akses posts API dengan parameter filter by userId
URL_POSTS = "https://jsonplaceholder.typicode.com/posts"
params    = {'userId': 1}

try:
    response_posts = requests.get(URL_POSTS, params=params, timeout=10)

    if response_posts.status_code == 200:
        posts_data = response_posts.json()
        df_posts   = pd.DataFrame(posts_data)
        print(f'Posts milik userId=1: {len(df_posts)} post')
        print(df_posts[['id', 'userId', 'title']].head())
    else:
        print(f'Error: {response_posts.status_code}')

except requests.exceptions.ConnectionError:
    print('Gagal terhubung ke API.')

Posts milik userId=1: 10 post
   id  userId                                              title
0   1       1  sunt aut facere repellat provident occaecati e...
1   2       1                                       qui est esse
2   3       1  ea molestias quasi exercitationem repellat qui...
3   4       1                               eum et est occaecati
4   5       1                                 nesciunt quas odio


**Temuan Langkah 11:**  
- Berhasil mengakses REST API JSONPlaceholder tanpa API key  
- Respons JSON di-*normalize* dengan `json_normalize()` menjadi DataFrame Pandas  
- Kolom nested seperti `address.city` otomatis diubah menjadi `address_city`  
- Berhasil memfilter posts berdasarkan `userId` menggunakan parameter query string

---
## Ringkasan Pipeline Data Cleaning

| Langkah | Aksi | Hasil |
|---------|------|-------|
| Load Dataset | pd.read_csv() | 130 baris, 7 kolom |
| Eksplorasi | info(), describe(), isnull() | Ditemukan missing, outlier, inkonsistensi |
| Hapus Duplikat | drop_duplicates() | 0 duplikat ditemukan |
| Normalisasi String | str.strip().str.title() + mapping | kota & kondisi terstandarisasi |
| Imputasi | median (numerik), modus (kategorik) | 0 missing values |
| Tangani Outlier | IQR Fence + clip() | Nilai ekstrem di-cap |
| Validasi | assert missing=0, duplikat=0 | LULUS |
| Ekspor | to_csv() | housing_clean.csv tersimpan |
| API | requests + json_normalize | 10 users, 10 posts berhasil dimuat |